# 06 - Modelagem de Tópicos e Clusterização (LDA, NMF e K-Means)

Este notebook carrega o corpus pré-processado e as stopwords curadas, gera as representações `CountVectorizer` e `TF-IDF`, ajusta os modelos **LDA**, **NMF** e **K-Means (com TruncatedSVD)** e exporta os termos principais de cada tópico/cluster em `results/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.curadoria import carregar_stopwords_curadas
from src.modelagem import (
    extrair_top_termos,
    treinar_kmeans_svd,
    treinar_lda,
    treinar_nmf,
)
from src.vetorizacao import carregar_corpus, vetorizar_count, vetorizar_tfidf

## 1. Parâmetros dos Modelos

In [ ]:
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_preprocessado.csv"
CURADORIA_FILE = PROJECT_ROOT / "data" / "processed" / "curadoria_stopwords.csv"
RESULTS_DIR = PROJECT_ROOT / "results"

COLUMN_ID = "id"
COLUMN_TEXT = "clean_text"

# Hiperparâmetros de vetorização
NGRAM_RANGE = (1, 1)
MAX_DF = 0.85
MIN_DF = 5

# Hiperparâmetros de modelagem
N_TOPICS = 5
N_COMPONENTS_SVD = 100
N_TOP_WORDS = 10
RANDOM_STATE = 42

## 2. Carregamento do Corpus e Vetorização

In [ ]:
df_corpus = carregar_corpus(INPUT_PATH, column_id=COLUMN_ID, column_text=COLUMN_TEXT)

if CURADORIA_FILE.exists():
    stopwords_final = carregar_stopwords_curadas(CURADORIA_FILE)
else:
    print("Arquivo de curadoria não encontrado; prosseguindo sem stopwords adicionais.")
    stopwords_final = set()

count_vectorizer, count_matrix, _ = vetorizar_count(
    df_corpus[COLUMN_TEXT],
    stop_words=stopwords_final,
    ngram_range=NGRAM_RANGE,
    max_df=MAX_DF,
    min_df=MIN_DF,
)

tfidf_vectorizer, tfidf_matrix, _ = vetorizar_tfidf(
    df_corpus[COLUMN_TEXT],
    stop_words=stopwords_final,
    ngram_range=NGRAM_RANGE,
    max_df=MAX_DF,
    min_df=MIN_DF,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 3. LDA (Latent Dirichlet Allocation)

In [ ]:
lda = treinar_lda(count_matrix, n_topics=N_TOPICS, random_state=RANDOM_STATE)

topicos_lda = extrair_top_termos(
    lda.components_,
    count_vectorizer.get_feature_names_out(),
    n_top_words=N_TOP_WORDS,
    prefixo="LDA Tópico",
)
topicos_lda.to_csv(RESULTS_DIR / "topicos_lda.csv", index=False, encoding="utf-8")
topicos_lda

## 4. NMF (Non-Negative Matrix Factorization)

In [ ]:
nmf, W, H = treinar_nmf(tfidf_matrix, n_topics=N_TOPICS, random_state=RANDOM_STATE)

topicos_nmf = extrair_top_termos(
    H,
    tfidf_vectorizer.get_feature_names_out(),
    n_top_words=N_TOP_WORDS,
    prefixo="NMF Tópico",
)
topicos_nmf.to_csv(RESULTS_DIR / "topicos_nmf.csv", index=False, encoding="utf-8")
topicos_nmf

## 5. K-Means com Redução de Dimensionalidade (TruncatedSVD)

In [ ]:
svd, tfidf_reduced, kmeans = treinar_kmeans_svd(
    tfidf_matrix,
    n_clusters=N_TOPICS,
    n_components_svd=N_COMPONENTS_SVD,
    random_state=RANDOM_STATE,
)

centroides_originais = svd.inverse_transform(kmeans.cluster_centers_)
clusters_kmeans = extrair_top_termos(
    centroides_originais,
    tfidf_vectorizer.get_feature_names_out(),
    n_top_words=N_TOP_WORDS,
    prefixo="KMeans Cluster",
)
clusters_kmeans.to_csv(RESULTS_DIR / "clusters_kmeans.csv", index=False, encoding="utf-8")
clusters_kmeans